# Missing Data Analysis: price_per_square_meter

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from scipy.stats import chi2_contingency, chi2

In [ ]:
df = pd.read_excel('../data/properties_sampled.xlsx')

target = 'price_per_square_meter'
missing_mask = df[target].isna()

print(f"Missing: {missing_mask.sum()} / {len(df)} ({missing_mask.mean():.1%})\n")

# MCAR: Little's test (simplified chi-square on missing patterns)
numeric = df.select_dtypes(include=np.number)
pattern_counts = numeric.isnull().value_counts().values
chi2_stat = np.sum((pattern_counts - pattern_counts.mean()) ** 2 / pattern_counts.mean())
p_mcar = 1 - chi2.cdf(chi2_stat, len(pattern_counts) - 1)
print(f"Little's MCAR test  →  chi2={chi2_stat:.2f}, p={p_mcar:.4f}  →  {'MCAR not rejected' if p_mcar > 0.05 else 'MCAR rejected'}\n")

# MAR: does missingness depend on observed variables?
results = []

cat_vars = [c for c in ['market', 'material', 'district', 'river_side'] if c in df.columns]
for var in cat_vars:
    ct = pd.crosstab(df[var].fillna('Missing'), missing_mask)
    chi2_s, p, *_ = chi2_contingency(ct)
    results.append((var, 'categorical', p))

num_vars = [c for c in ['surface', 'rooms', 'floor', 'construction_year'] if c in df.columns]
for var in num_vars:
    g0 = df.loc[~missing_mask, var].dropna()
    g1 = df.loc[missing_mask, var].dropna()
    if len(g0) and len(g1):
        _, p = stats.ttest_ind(g0, g1) # t-test assumes normality and equal variances, which may not hold. I belive it could be worth to consider non-parametric tests if needed.
        results.append((var, 'numerical', p))

print(f"{'Variable':<25} {'Type':<12} {'p-value':<10} Significant?")
print("-" * 55)
for var, vtype, p in results:
    sig = "YES ***" if p < 0.05 else "no"
    print(f"{var:<25} {vtype:<12} {p:<10.4f} {sig}")

n_sig = sum(1 for *_, p in results if p < 0.05)
print(f"\n{n_sig}/{len(results)} variables significantly related to missingness.")


## Conclusion

- **MCAR rejected** → missingness is not completely random.
- **MAR**: missingness depends on observed variables (market, material, construction year) → data is likely **MAR**.
- **MNAR** cannot be ruled out (prices may be withheld intentionally for high/low-value properties).

**My approach**: I will use use multiple imputation (MICE).


#### **Area for improvment**: It could be worth to replace t-test with a non-parametric method as the distribution of features is not normal